# Green Taxi Bronze Data Quality Validation

This notebook validates the Green Taxi Bronze dataset before Silver processing.

Bronze table:

`ftw-week-08`.`02-bronze`.green_taxi_raw

DQ results table:

`ftw-week-08`.`02-bronze`.90_validate_green_taxi

Source period:

March–May 2026

Source files:

- `green_tripdata_2026-03.parquet`
- `green_tripdata_2026-04.parquet`
- `green_tripdata_2026-05.parquet`

Bronze preserves the source values. Source-level anomalies are measured and documented rather than silently modified.

# 1. Validation Approach

The Bronze validation covers:

- Volume
- Schema
- Required timestamps
- Timestamp consistency
- Reporting-period coverage
- Passenger-count conditions
- Trip-distance validity
- Fare and total-amount conditions
- Source categorical domains
- Location ID validity
- Full-row duplicates
- Provenance completeness

The source-to-Bronze row-count reconciliation is performed separately.

Status meanings:

- `PASS` — expectation is satisfied
- `WARN` — non-blocking source-quality condition
- `FAIL` — blocking Bronze issue
- `INFO` — measurement only

WARN and INFO results do not block the Bronze DQ gate.

# 2. Dataset and Source Context

The Green Taxi source consists of three monthly Parquet files covering March–May 2026.

Expected source row counts:

| Source file | Expected rows |
|---|---:|
| `green_tripdata_2026-03.parquet` | 44,208 |
| `green_tripdata_2026-04.parquet` | 44,238 |
| `green_tripdata_2026-05.parquet` | 44,921 |
| **Total** | **133,367** |

There is no natural trip identifier in the source. Therefore, duplicate validation uses the complete set of source business fields rather than treating `VendorID` as a trip key.

Bronze preserves the source fields and values. Silver owns standardization and objective quality flags.

# 3. Bronze Profile

In [0]:
%sql
-- ============================================================
-- BRONZE PROFILE
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT source_file) AS source_file_count,
    MIN(lpep_pickup_datetime) AS earliest_pickup,
    MAX(lpep_pickup_datetime) AS latest_pickup
FROM `ftw-week-08`.`02-bronze`.green_taxi_raw;

SELECT
    source_file,
    COUNT(*) AS row_count,
    MIN(lpep_pickup_datetime) AS earliest_pickup,
    MAX(lpep_pickup_datetime) AS latest_pickup
FROM `ftw-week-08`.`02-bronze`.green_taxi_raw
GROUP BY source_file
ORDER BY source_file;

In [0]:
%sql
-- ============================================================
-- BRONZE SCHEMA PROFILE
-- ============================================================

DESCRIBE `ftw-week-08`.`02-bronze`.green_taxi_raw;

In [0]:
%sql
-- ============================================================
-- CREATE DQ RESULTS TABLE
-- ============================================================

CREATE TABLE IF NOT EXISTS `ftw-week-08`.`02-bronze`.90_validate_green_taxi (
    run_id STRING,
    executed_at TIMESTAMP,
    layer STRING,
    dataset STRING,
    batch_id STRING,
    source_version_id STRING,
    code_revision STRING,
    check_name STRING,
    check_type STRING,
    status STRING,
    severity STRING,
    fail_count BIGINT,
    total_count BIGINT,
    fail_pct DOUBLE,
    threshold_pct DOUBLE,
    metric_value DOUBLE,
    owner STRING,
    details STRING,
    evidence_location STRING
)
USING DELTA;

In [0]:
%sql
-- ============================================================
-- RUN CONTEXT
-- Generate one unique run_id for this validation run.
-- ============================================================

DECLARE OR REPLACE VARIABLE dq_run_id STRING;

SET VAR dq_run_id = uuid();

SELECT dq_run_id AS run_id;

In [0]:
%sql
-- ============================================================
-- GREEN TAXI BRONZE DQ EXECUTION
-- One execution creates exactly one result row per check.
-- ============================================================

DELETE FROM `ftw-week-08`.`02-bronze`.90_validate_green_taxi
WHERE run_id = dq_run_id
  AND layer = 'Bronze'
  AND dataset = 'green_taxi';

INSERT INTO `ftw-week-08`.`02-bronze`.90_validate_green_taxi

WITH total AS (
    SELECT COUNT(*) AS total_count
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw
),

schema_expected AS (
    SELECT *
    FROM VALUES
        ('VendorID', 'INT', 0),
        ('lpep_pickup_datetime', 'TIMESTAMP_NTZ', 1),
        ('lpep_dropoff_datetime', 'TIMESTAMP_NTZ', 2),
        ('store_and_fwd_flag', 'STRING', 3),
        ('RatecodeID', 'LONG', 4),
        ('PULocationID', 'INT', 5),
        ('DOLocationID', 'INT', 6),
        ('passenger_count', 'LONG', 7),
        ('trip_distance', 'DOUBLE', 8),
        ('fare_amount', 'DOUBLE', 9),
        ('extra', 'DOUBLE', 10),
        ('mta_tax', 'DOUBLE', 11),
        ('tip_amount', 'DOUBLE', 12),
        ('tolls_amount', 'DOUBLE', 13),
        ('ehail_fee', 'DOUBLE', 14),
        ('improvement_surcharge', 'DOUBLE', 15),
        ('total_amount', 'DOUBLE', 16),
        ('payment_type', 'LONG', 17),
        ('trip_type', 'LONG', 18),
        ('congestion_surcharge', 'DOUBLE', 19),
        ('cbd_congestion_fee', 'DOUBLE', 20),
        ('source_system', 'STRING', 21),
        ('source_file', 'STRING', 22),
        ('ingested_at', 'TIMESTAMP', 23),
        ('batch_id', 'STRING', 24)
    AS expected(column_name, data_type, ordinal_position)
),

schema_actual AS (
    SELECT
        column_name,
        data_type,
        ordinal_position
    FROM `system`.information_schema.columns
    WHERE table_catalog = 'ftw-week-08'
      AND table_schema = '02-bronze'
      AND table_name = 'green_taxi_raw'
),

schema_mismatches AS (
    SELECT
        COUNT(*) AS fail_count
    FROM (
        SELECT
            expected.column_name,
            expected.data_type,
            expected.ordinal_position
        FROM schema_expected expected

        FULL OUTER JOIN schema_actual actual
            ON expected.column_name = actual.column_name
           AND expected.data_type = actual.data_type
           AND expected.ordinal_position = actual.ordinal_position

        WHERE expected.column_name IS NULL
           OR actual.column_name IS NULL
    )
),

duplicate_rows AS (
    SELECT
        SUM(duplicate_count - 1) AS fail_count
    FROM (
        SELECT
            VendorID,
            lpep_pickup_datetime,
            lpep_dropoff_datetime,
            store_and_fwd_flag,
            RatecodeID,
            PULocationID,
            DOLocationID,
            passenger_count,
            trip_distance,
            fare_amount,
            extra,
            mta_tax,
            tip_amount,
            tolls_amount,
            ehail_fee,
            improvement_surcharge,
            total_amount,
            payment_type,
            trip_type,
            congestion_surcharge,
            cbd_congestion_fee,
            COUNT(*) AS duplicate_count
        FROM `ftw-week-08`.`02-bronze`.green_taxi_raw
        GROUP BY
            VendorID,
            lpep_pickup_datetime,
            lpep_dropoff_datetime,
            store_and_fwd_flag,
            RatecodeID,
            PULocationID,
            DOLocationID,
            passenger_count,
            trip_distance,
            fare_amount,
            extra,
            mta_tax,
            tip_amount,
            tolls_amount,
            ehail_fee,
            improvement_surcharge,
            total_amount,
            payment_type,
            trip_type,
            congestion_surcharge,
            cbd_congestion_fee
        HAVING COUNT(*) > 1
    )
),

checks AS (

    -- ========================================================
    -- 1. ROW COUNT
    -- ========================================================
    SELECT
        'row_count_not_empty' AS check_name,
        'VOLUME' AS check_type,
        'FAIL' AS severity,
        0.0 AS threshold_pct,
        CASE
            WHEN COUNT(*) = 0 THEN 1
            ELSE 0
        END AS fail_count,
        CAST(NULL AS DOUBLE) AS metric_value,
        'Bronze table must contain rows.' AS details
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 2. SOURCE SCHEMA
    -- ========================================================
    SELECT
        'source_schema',
        'SCHEMA',
        'FAIL',
        0.0,
        CAST(fail_count AS BIGINT),
        CAST(NULL AS DOUBLE),
        'Bronze schema must match the expected 25-column structure and Databricks data types.'
    FROM schema_mismatches

    UNION ALL

    -- ========================================================
    -- 3. PICKUP TIMESTAMP NOT NULL
    -- ========================================================
    SELECT
        'pickup_timestamp_not_null',
        'NOT_NULL',
        'FAIL',
        0.0,
        SUM(
            CASE
                WHEN lpep_pickup_datetime IS NULL THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Pickup timestamp is required.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 4. DROPOFF TIMESTAMP NOT NULL
    -- ========================================================
    SELECT
        'dropoff_timestamp_not_null',
        'NOT_NULL',
        'FAIL',
        0.0,
        SUM(
            CASE
                WHEN lpep_dropoff_datetime IS NULL THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Dropoff timestamp is required.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 5. DROPOFF AFTER PICKUP
    -- ========================================================
    SELECT
        'dropoff_after_pickup',
        'CONSISTENCY',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN lpep_pickup_datetime IS NOT NULL
                 AND lpep_dropoff_datetime IS NOT NULL
                 AND lpep_dropoff_datetime <= lpep_pickup_datetime
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Dropoff should be strictly later than pickup. Anomalies are retained for Silver handling.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 6. REPORTING PERIOD COVERAGE
    -- ========================================================
    SELECT
        'expected_month_coverage',
        'DATE_COVERAGE',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN lpep_pickup_datetime < TIMESTAMP('2026-03-01 00:00:00')
                  OR lpep_pickup_datetime >= TIMESTAMP('2026-06-01 00:00:00')
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Pickup timestamps outside the March–May 2026 reporting period are retained and flagged.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 7. PASSENGER COUNT > 8
    -- ========================================================
    SELECT
        'passenger_count_gt_8',
        'RANGE',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN passenger_count > 8 THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Passenger counts greater than 8 are separately flagged.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 8. PASSENGER COUNT = 0
    -- ========================================================
    SELECT
        'passenger_count_zero',
        'MEASURE',
        'WARN',
        NULL,
        SUM(
            CASE
                WHEN passenger_count = 0 THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Zero passenger counts are retained and separately flagged for Silver handling.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 9. PASSENGER COUNT NULL
    -- ========================================================
    SELECT
        'passenger_count_null',
        'MEASURE',
        'WARN',
        NULL,
        SUM(
            CASE
                WHEN passenger_count IS NULL THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Null passenger counts are retained. No Bronze imputation is performed.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 10. TRIP DISTANCE NON-NEGATIVE
    -- ========================================================
    SELECT
        'trip_distance_non_negative',
        'RANGE',
        'FAIL',
        0.0,
        SUM(
            CASE
                WHEN trip_distance < 0 THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Trip distance must be zero or greater.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 11. NEGATIVE FARE
    -- ========================================================
    SELECT
        'negative_fare_amount',
        'MEASURE',
        'WARN',
        NULL,
        SUM(
            CASE
                WHEN fare_amount < 0 THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Negative fare values are retained and flagged for downstream handling.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 12. NEGATIVE TOTAL
    -- ========================================================
    SELECT
        'negative_total_amount',
        'MEASURE',
        'WARN',
        NULL,
        SUM(
            CASE
                WHEN total_amount < 0 THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Negative total values are retained and flagged for downstream handling.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 13. ZERO DISTANCE / HIGH FARE
    -- ========================================================
    SELECT
        'zero_distance_high_fare',
        'MEASURE',
        'INFO',
        NULL,
        SUM(
            CASE
                WHEN trip_distance = 0
                 AND fare_amount > 20
                THEN 1
                ELSE 0
            END
        ),
        CAST(
            SUM(
                CASE
                    WHEN trip_distance = 0
                     AND fare_amount > 20
                    THEN 1
                    ELSE 0
                END
            ) AS DOUBLE
        ),
        'Sensitivity-review condition. Rows remain in Bronze.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 14. VENDOR ID DOMAIN
    -- ========================================================
    SELECT
        'vendor_id_domain',
        'DOMAIN',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN VendorID IS NOT NULL
                 AND VendorID NOT IN (1, 2, 6)
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Valid profiled VendorID values are 1, 2, and 6.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 15. PAYMENT TYPE DOMAIN
    -- ========================================================
    SELECT
        'payment_type_domain',
        'DOMAIN',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN payment_type IS NOT NULL
                 AND payment_type NOT IN (0, 1, 2, 3, 4, 5, 6)
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Valid profiled payment_type values are 0–6.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 16. RATECODE DOMAIN
    -- ========================================================
    SELECT
        'ratecode_id_domain',
        'DOMAIN',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN RatecodeID IS NOT NULL
                 AND RatecodeID NOT IN (1, 2, 3, 4, 5, 6, 99)
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Valid profiled RatecodeID values are 1–6 and 99.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 17. PICKUP LOCATION ID RANGE
    -- ========================================================
    SELECT
        'pickup_location_id_range',
        'RANGE',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN PULocationID IS NULL
                  OR PULocationID < 1
                  OR PULocationID > 265
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Pickup LocationID must be within the Taxi Zone snapshot range 1–265.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 18. DROPOFF LOCATION ID RANGE
    -- ========================================================
    SELECT
        'dropoff_location_id_range',
        'RANGE',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN DOLocationID IS NULL
                  OR DOLocationID < 1
                  OR DOLocationID > 265
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Dropoff LocationID must be within the Taxi Zone snapshot range 1–265.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 19. FULL SOURCE ROW DUPLICATES
    -- ========================================================
    SELECT
        'full_source_row_duplicate',
        'UNIQUE',
        'WARN',
        0.01,
        CAST(COALESCE(fail_count, 0) AS BIGINT),
        CAST(NULL AS DOUBLE),
        'Duplicate source rows are measured using all source business fields. There is no natural trip ID.'
    FROM duplicate_rows

    UNION ALL

    -- ========================================================
    -- 20. PROVENANCE COMPLETENESS
    -- ========================================================
    SELECT
        'provenance_completeness',
        'PROVENANCE',
        'FAIL',
        0.0,
        SUM(
            CASE
                WHEN source_system IS NULL
                  OR source_file IS NULL
                  OR ingested_at IS NULL
                  OR batch_id IS NULL
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'Required Bronze provenance fields must be populated.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

    UNION ALL

    -- ========================================================
    -- 21. SOURCE SYSTEM DOMAIN
    -- ========================================================
    SELECT
        'source_system_domain',
        'DOMAIN',
        'WARN',
        0.01,
        SUM(
            CASE
                WHEN source_system IS NULL
                  OR source_system <> 'nyc_tlc_green'
                THEN 1
                ELSE 0
            END
        ),
        CAST(NULL AS DOUBLE),
        'source_system should identify the NYC TLC Green Taxi source.'
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw
)

SELECT
    dq_run_id AS run_id,
    current_timestamp() AS executed_at,
    'Bronze' AS layer,
    'green_taxi' AS dataset,
    CAST(NULL AS STRING) AS batch_id,
    CAST(NULL AS STRING) AS source_version_id,
    CAST(NULL AS STRING) AS code_revision,
    c.check_name,
    c.check_type,

    CASE
        WHEN c.severity = 'INFO' THEN 'INFO'
        WHEN c.fail_count = 0 THEN 'PASS'
        WHEN c.severity = 'FAIL' THEN 'FAIL'
        WHEN c.threshold_pct IS NOT NULL
         AND c.fail_count / NULLIF(t.total_count, 0) <= c.threshold_pct
            THEN 'WARN'
        ELSE 'WARN'
    END AS status,

    c.severity,
    CAST(c.fail_count AS BIGINT) AS fail_count,
    t.total_count,
    100.0 * c.fail_count / NULLIF(t.total_count, 0) AS fail_pct,
    c.threshold_pct,
    c.metric_value,
    'Data Engineering' AS owner,
    c.details,
    CAST(NULL AS STRING) AS evidence_location

FROM checks c
CROSS JOIN total t;

In [0]:
%sql
-- ============================================================
-- REVIEW CURRENT DQ RUN
-- ============================================================

SELECT
    check_name,
    check_type,
    status,
    severity,
    fail_count,
    total_count,
    ROUND(fail_pct, 4) AS fail_pct,
    threshold_pct,
    metric_value,
    details
FROM `ftw-week-08`.`02-bronze`.90_validate_green_taxi
WHERE run_id = dq_run_id
ORDER BY
    CASE status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
        WHEN 'PASS' THEN 3
        WHEN 'INFO' THEN 4
        ELSE 5
    END,
    check_name;

In [0]:
%sql
-- ============================================================
-- DQ SUMMARY
-- ============================================================

SELECT
    status,
    COUNT(*) AS check_count
FROM `ftw-week-08`.`02-bronze`.90_validate_green_taxi
WHERE run_id = dq_run_id
GROUP BY status
ORDER BY
    CASE status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
        WHEN 'PASS' THEN 3
        WHEN 'INFO' THEN 4
        ELSE 5
    END;

In [0]:
%sql
-- ============================================================
-- BRONZE DQ EXIT GATE
-- ============================================================

SELECT
    CASE
        WHEN SUM(
            CASE
                WHEN status = 'FAIL' THEN 1
                ELSE 0
            END
        ) = 0
        THEN 'READY_FOR_SILVER'
        ELSE 'BLOCKED'
    END AS bronze_dq_gate,

    SUM(
        CASE
            WHEN status = 'FAIL' THEN 1
            ELSE 0
        END
    ) AS fail_count,

    SUM(
        CASE
            WHEN status = 'WARN' THEN 1
            ELSE 0
        END
    ) AS warn_count,

    SUM(
        CASE
            WHEN status = 'PASS' THEN 1
            ELSE 0
        END
    ) AS pass_count,

    SUM(
        CASE
            WHEN status = 'INFO' THEN 1
            ELSE 0
        END
    ) AS info_count

FROM `ftw-week-08`.`02-bronze`.90_validate_green_taxi
WHERE run_id = dq_run_id;

# 4. Source → Bronze Reconciliation

In [0]:
%sql
-- ============================================================
-- SOURCE → BRONZE RECONCILIATION
-- ============================================================

WITH expected AS (

    SELECT
        'green_tripdata_2026-03.parquet' AS source_file,
        44208 AS expected_rows

    UNION ALL

    SELECT
        'green_tripdata_2026-04.parquet',
        44238

    UNION ALL

    SELECT
        'green_tripdata_2026-05.parquet',
        44921
),

actual AS (

    SELECT
        source_file,
        COUNT(*) AS actual_rows
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw
    GROUP BY source_file
)

SELECT
    e.source_file,
    e.expected_rows,
    COALESCE(a.actual_rows, 0) AS actual_rows,

    COALESCE(a.actual_rows, 0) - e.expected_rows
        AS row_difference,

    CASE
        WHEN COALESCE(a.actual_rows, 0) = e.expected_rows
        THEN 'PASS'
        ELSE 'FAIL'
    END AS reconciliation_status

FROM expected e

LEFT JOIN actual a
    ON e.source_file = a.source_file

ORDER BY e.source_file;

In [0]:
%sql
-- ============================================================
-- OVERALL SOURCE → BRONZE ROW COUNT
-- ============================================================

WITH expected AS (
    SELECT 44208 AS expected_rows
    UNION ALL
    SELECT 44238
    UNION ALL
    SELECT 44921
),

actual AS (
    SELECT COUNT(*) AS actual_rows
    FROM `ftw-week-08`.`02-bronze`.green_taxi_raw
)

SELECT
    SUM(expected_rows) AS expected_total_rows,
    actual_rows,
    actual_rows - SUM(expected_rows) AS row_difference,

    CASE
        WHEN actual_rows = SUM(expected_rows)
        THEN 'PASS'
        ELSE 'FAIL'
    END AS reconciliation_status

FROM expected
CROSS JOIN actual
GROUP BY actual_rows;

# 5. WARN / FAIL Investigation

In [0]:
%sql
-- ============================================================
-- WARN / FAIL INVESTIGATION
-- ============================================================

SELECT
    source_file,
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN fare_amount < 0 THEN 1
            ELSE 0
        END
    ) AS negative_fare_rows,

    SUM(
        CASE
            WHEN total_amount < 0 THEN 1
            ELSE 0
        END
    ) AS negative_total_rows,

    SUM(
        CASE
            WHEN trip_distance = 0
             AND fare_amount > 20
            THEN 1
            ELSE 0
        END
    ) AS zero_distance_high_fare_rows,

    SUM(
        CASE
            WHEN passenger_count = 0 THEN 1
            ELSE 0
        END
    ) AS passenger_count_zero_rows,

    SUM(
        CASE
            WHEN passenger_count > 8 THEN 1
            ELSE 0
        END
    ) AS passenger_count_gt_8_rows,

    SUM(
        CASE
            WHEN passenger_count IS NULL THEN 1
            ELSE 0
        END
    ) AS passenger_count_null_rows,

    SUM(
        CASE
            WHEN lpep_dropoff_datetime <= lpep_pickup_datetime
            THEN 1
            ELSE 0
        END
    ) AS invalid_duration_rows,

    SUM(
        CASE
            WHEN lpep_pickup_datetime < TIMESTAMP('2026-03-01 00:00:00')
              OR lpep_pickup_datetime >= TIMESTAMP('2026-06-01 00:00:00')
            THEN 1
            ELSE 0
        END
    ) AS out_of_period_rows

FROM `ftw-week-08`.`02-bronze`.green_taxi_raw

GROUP BY source_file

ORDER BY source_file;

# 6. Final Review

Known source-level conditions include:

- Negative `fare_amount`
- Negative `total_amount`
- Zero-distance trips with higher fares
- `passenger_count = 0`
- `passenger_count > 8`
- Null `passenger_count`
- A small number of pickup timestamps outside the March–May 2026 reporting window

These conditions are preserved in Bronze.

Silver should apply the approved quality flags and measure-specific eligibility rules rather than silently modifying the Bronze source data.

The Bronze DQ gate is:

- `READY_FOR_SILVER` when there are zero FAIL results
- `BLOCKED` when one or more FAIL results remain

INFO results are measurements only and do not block the Bronze gate.